# 🌾 Dashboard Agrícola Argentina (1981–2024)

Visión general de la producción agrícola argentina: superficie sembrada/cosechada,
producción, rendimiento y variables climáticas asociadas.

**Fuente:** `df_recortado.csv` — 111.417 registros, 40 cultivos, 23 provincias.

> Ejecutá las celdas en orden (▶️ Runtime → Run all). Si estás en Colab, subí el
> archivo `df_recortado.csv` cuando se te pida en la primera celda de carga.


In [ ]:

# 📦 Instalación de librerías (Colab ya trae la mayoría)
!pip install plotly -q


In [ ]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)


## 1. Carga de datos

Si estás en Google Colab y el archivo no está en el entorno, se te va a pedir que
lo subas manualmente. Si ya lo tenés en tu Drive, comentá la sección de `files.upload()`
y usá la ruta correspondiente.


In [ ]:

import os

CSV_PATH = "df_recortado.csv"

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print("Subí el archivo df_recortado.csv:")
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            "No se encontró df_recortado.csv. Colocalo en el mismo directorio "
            "que este notebook o subilo manualmente."
        )

df = pd.read_csv(CSV_PATH)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()


## 2. Panorama general del dataset

In [ ]:

print("Rango de años:", df['año'].min(), "-", df['año'].max())
print("Cultivos:", df['cultivo'].nunique())
print("Provincias:", df['provincia'].nunique())
print("Departamentos:", df['departamento'].nunique())
print("\nValores nulos por columna:")
print(df.isnull().sum().sum(), "nulos en total")
df.describe()


In [ ]:

# KPIs principales
total_prod = df['produccion_tm'].sum()
total_sup_sembrada = df['superficie_sembrada_ha'].sum()
total_sup_cosechada = df['superficie_cosechada_ha'].sum()
rendimiento_prom = df['rendimiento_kgxha'].mean()

fig = go.Figure()
kpis = [
    ("Producción total (Tm)", f"{total_prod/1e6:,.1f} M"),
    ("Superficie sembrada (ha)", f"{total_sup_sembrada/1e6:,.1f} M"),
    ("Superficie cosechada (ha)", f"{total_sup_cosechada/1e6:,.1f} M"),
    ("Rendimiento promedio (kg/ha)", f"{rendimiento_prom:,.0f}"),
]
fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}]*4])
for i, (label, value) in enumerate(kpis):
    fig.add_trace(go.Indicator(mode="number", value=None, title={"text": label}), row=1, col=i+1)

# Plotly Indicator necesita valores numéricos; mostramos como tabla simple en su lugar
import plotly.figure_factory as ff
kpi_df = pd.DataFrame(kpis, columns=["Métrica", "Valor"])
fig2 = ff.create_table(kpi_df)
fig2.update_layout(title="📊 KPIs generales del dataset")
fig2.show()


## 3. Evolución temporal (1981–2024)

Cómo evolucionaron la superficie sembrada, la superficie cosechada y la producción
total a lo largo del tiempo, sumando todos los cultivos y provincias.


In [ ]:

evol = df.groupby('año').agg(
    superficie_sembrada_ha=('superficie_sembrada_ha', 'sum'),
    superficie_cosechada_ha=('superficie_cosechada_ha', 'sum'),
    produccion_tm=('produccion_tm', 'sum'),
).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=evol['año'], y=evol['superficie_sembrada_ha'],
                          name='Superficie sembrada (ha)', mode='lines'), secondary_y=False)
fig.add_trace(go.Scatter(x=evol['año'], y=evol['superficie_cosechada_ha'],
                          name='Superficie cosechada (ha)', mode='lines'), secondary_y=False)
fig.add_trace(go.Bar(x=evol['año'], y=evol['produccion_tm'],
                      name='Producción (Tm)', opacity=0.3), secondary_y=True)

fig.update_layout(title="Evolución de superficie y producción total por año",
                   xaxis_title="Año", legend_title="Serie", template="plotly_white")
fig.update_yaxes(title_text="Hectáreas", secondary_y=False)
fig.update_yaxes(title_text="Producción (Tm)", secondary_y=True)
fig.show()


In [ ]:

rend_evol = df.groupby('año')['rendimiento_kgxha'].mean().reset_index()

fig = px.line(rend_evol, x='año', y='rendimiento_kgxha',
              title="Evolución del rendimiento promedio (kg/ha) por año",
              template="plotly_white", markers=True)
fig.update_layout(xaxis_title="Año", yaxis_title="Rendimiento promedio (kg/ha)")
fig.show()


## 4. Ranking de cultivos

In [ ]:

top_cultivos_prod = (df.groupby('cultivo')['produccion_tm'].sum()
                        .sort_values(ascending=False).head(10).reset_index())

fig = px.bar(top_cultivos_prod, x='produccion_tm', y='cultivo', orientation='h',
             title="Top 10 cultivos por producción total acumulada (Tm)",
             template="plotly_white", color='produccion_tm',
             color_continuous_scale='Greens')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title="Producción (Tm)", yaxis_title="")
fig.show()


In [ ]:

top_cultivos_sup = (df.groupby('cultivo')['superficie_sembrada_ha'].sum()
                       .sort_values(ascending=False).head(10).reset_index())

fig = px.bar(top_cultivos_sup, x='superficie_sembrada_ha', y='cultivo', orientation='h',
             title="Top 10 cultivos por superficie sembrada acumulada (ha)",
             template="plotly_white", color='superficie_sembrada_ha',
             color_continuous_scale='Oranges')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title="Superficie sembrada (ha)", yaxis_title="")
fig.show()


In [ ]:

# Rendimiento promedio por cultivo (solo cultivos con suficientes registros)
min_registros = 100
rend_cultivo = (df.groupby('cultivo').filter(lambda g: len(g) >= min_registros)
                   .groupby('cultivo')['rendimiento_kgxha'].mean()
                   .sort_values(ascending=False).head(15).reset_index())

fig = px.bar(rend_cultivo, x='rendimiento_kgxha', y='cultivo', orientation='h',
             title="Top 15 cultivos por rendimiento promedio (kg/ha)",
             template="plotly_white", color='rendimiento_kgxha',
             color_continuous_scale='Blues')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title="Rendimiento promedio (kg/ha)", yaxis_title="")
fig.show()


## 5. Distribución geográfica

In [ ]:

prod_provincia = (df.groupby('provincia')['produccion_tm'].sum()
                     .sort_values(ascending=False).reset_index())

fig = px.bar(prod_provincia, x='provincia', y='produccion_tm',
             title="Producción total acumulada por provincia (Tm)",
             template="plotly_white", color='produccion_tm',
             color_continuous_scale='Viridis')
fig.update_layout(xaxis_title="", yaxis_title="Producción (Tm)", xaxis_tickangle=-45)
fig.show()


In [ ]:

# Mapa de calor: top cultivos x top provincias (superficie sembrada)
top_c = top_cultivos_sup['cultivo'].tolist()
top_p = prod_provincia['provincia'].head(10).tolist()

pivot = (df[df['cultivo'].isin(top_c) & df['provincia'].isin(top_p)]
         .pivot_table(index='cultivo', columns='provincia',
                       values='superficie_sembrada_ha', aggfunc='sum', fill_value=0))
pivot = pivot.loc[top_c, top_p]

fig = px.imshow(pivot, text_auto='.2s', aspect='auto',
                 title="Superficie sembrada (ha): top cultivos x top provincias",
                 color_continuous_scale='YlGnBu')
fig.update_layout(xaxis_title="Provincia", yaxis_title="Cultivo")
fig.show()


## 6. Variables climáticas y su relación con el rendimiento

In [ ]:

clima_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'precip_total_mm']
corr = df[clima_cols + ['rendimiento_kgxha']].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                 title="Correlación entre variables climáticas y rendimiento")
fig.show()


In [ ]:

sample = df.sample(min(5000, len(df)), random_state=42)

fig = px.scatter(sample, x='precip_total_mm', y='rendimiento_kgxha',
                  color='cultivo', opacity=0.5,
                  title="Precipitación total vs. rendimiento (muestra de 5.000 registros)",
                  template="plotly_white",
                  labels={'precip_total_mm': 'Precipitación total (mm)',
                          'rendimiento_kgxha': 'Rendimiento (kg/ha)'})
fig.update_layout(showlegend=False)
fig.show()


In [ ]:

temp_evol = df.groupby('año')[['T2M', 'T2M_MAX', 'T2M_MIN']].mean().reset_index()

fig = go.Figure()
for col, label in [('T2M', 'Temp. promedio'), ('T2M_MAX', 'Temp. máxima'), ('T2M_MIN', 'Temp. mínima')]:
    fig.add_trace(go.Scatter(x=temp_evol['año'], y=temp_evol[col], name=label, mode='lines'))

fig.update_layout(title="Evolución de la temperatura promedio anual (°C)",
                   xaxis_title="Año", yaxis_title="Temperatura (°C)", template="plotly_white")
fig.show()


## 7. Explorá vos mismo/a 🔍

Filtrá por cultivo y provincia para ver la evolución de la producción en detalle.
Cambiá los valores de `CULTIVO_FILTRO` y `PROVINCIA_FILTRO` (o dejá `None` para "todos").


In [ ]:

CULTIVO_FILTRO = "soja total"      # ej: "maíz", "trigo total", None para todos
PROVINCIA_FILTRO = None            # ej: "Buenos Aires", None para todas

sub = df.copy()
if CULTIVO_FILTRO:
    sub = sub[sub['cultivo'] == CULTIVO_FILTRO]
if PROVINCIA_FILTRO:
    sub = sub[sub['provincia'] == PROVINCIA_FILTRO]

evol_filtro = sub.groupby('año').agg(
    produccion_tm=('produccion_tm', 'sum'),
    rendimiento_kgxha=('rendimiento_kgxha', 'mean'),
).reset_index()

titulo = f"{CULTIVO_FILTRO or 'Todos los cultivos'} — {PROVINCIA_FILTRO or 'Todas las provincias'}"

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=evol_filtro['año'], y=evol_filtro['produccion_tm'],
                      name='Producción (Tm)', opacity=0.5), secondary_y=False)
fig.add_trace(go.Scatter(x=evol_filtro['año'], y=evol_filtro['rendimiento_kgxha'],
                          name='Rendimiento (kg/ha)', mode='lines+markers'), secondary_y=True)
fig.update_layout(title=titulo, xaxis_title="Año", template="plotly_white")
fig.update_yaxes(title_text="Producción (Tm)", secondary_y=False)
fig.update_yaxes(title_text="Rendimiento (kg/ha)", secondary_y=True)
fig.show()
